In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load dataset
df = pd.read_csv("WELFake_Dataset.csv")

# Drop first column (ID column)
df = df.iloc[:, 1:]

# Drop missing values
df = df.dropna()

# Combine title and text
df["content"] = df["title"] + " " + df["text"]

X = df["content"]

# Flip labels: 0 -> real, 1 -> fake
# Original: 0=fake, 1=real
y = df["label"].apply(lambda x: 1 - x)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# TF-IDF vectorization
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_df=0.7,
    min_df=2,
    ngram_range=(1, 2)
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Train Linear SVM
# LinearSVC is fast but has no predict_proba -> calibrate to get probabilities
base_svm = LinearSVC(class_weight="balanced", random_state=42)
svm = CalibratedClassifierCV(base_svm, method="sigmoid", cv=3)  # cv=3 keeps it fast

svm.fit(X_train_tfidf, y_train)

# Evaluate model on test set
y_pred = svm.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=["Real (0)", "Fake (1)"]))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Evaluate on training set
y_train_pred = svm.predict(X_train_tfidf)

print("\nTRAINING SET PERFORMANCE")
print("Accuracy:", accuracy_score(y_train, y_train_pred))
print(classification_report(y_train, y_train_pred, target_names=["Real (0)", "Fake (1)"]))

print("\nTEST SET PERFORMANCE")
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=["Real (0)", "Fake (1)"]))

print("\nTRAIN confusion matrix:")
print(confusion_matrix(y_train, y_train_pred))

print("\nTEST confusion matrix:")
print(confusion_matrix(y_test, y_pred))

#exec time 1.02

Accuracy: 0.9719038300251608
              precision    recall  f1-score   support

    Real (0)       0.97      0.97      0.97      7302
    Fake (1)       0.97      0.97      0.97      7006

    accuracy                           0.97     14308
   macro avg       0.97      0.97      0.97     14308
weighted avg       0.97      0.97      0.97     14308

Confusion Matrix:
[[7110  192]
 [ 210 6796]]

TRAINING SET PERFORMANCE
Accuracy: 0.999178738052386
              precision    recall  f1-score   support

    Real (0)       1.00      1.00      1.00     29207
    Fake (1)       1.00      1.00      1.00     28022

    accuracy                           1.00     57229
   macro avg       1.00      1.00      1.00     57229
weighted avg       1.00      1.00      1.00     57229


TEST SET PERFORMANCE
Accuracy: 0.9719038300251608
              precision    recall  f1-score   support

    Real (0)       0.97      0.97      0.97      7302
    Fake (1)       0.97      0.97      0.97      7006

   

In [23]:
# Crisis categories with associated keywords
CRISIS_CATEGORIES = {
    'health_crisis': [
        'pandemic', 'outbreak', 'epidemic', 'virus', 'disease', 'infection',
        'quarantine', 'lockdown', 'vaccine', 'death toll', 'hospitalizations',
        'emergency', 'covid', 'ebola', 'flu', 'health emergency', 'contagious',
        'symptoms', 'diagnosis', 'medical emergency', 'public health'
    ],
    'natural_disaster': [
        'earthquake', 'hurricane', 'tsunami', 'flood', 'wildfire', 'tornado',
        'cyclone', 'drought', 'avalanche', 'landslide', 'volcano', 'storm',
        'evacuation', 'disaster zone', 'emergency relief', 'natural disaster',
        'typhoon', 'blizzard', 'heatwave', 'mudslide'
    ],
    'political_crisis': [
        'coup', 'impeachment', 'scandal', 'resignation', 'protest', 'riot',
        'unrest', 'revolution', 'martial law', 'state of emergency', 'uprising',
        'government collapse', 'political turmoil', 'crisis talks', 'regime',
        'dictatorship', 'corruption', 'election fraud', 'political violence'
    ],
    'economic_crisis': [
        'recession', 'crash', 'bankruptcy', 'inflation', 'market collapse',
        'financial crisis', 'stock market', 'economic downturn', 'debt crisis',
        'unemployment', 'layoffs', 'economic emergency', 'depression',
        'foreclosure', 'bailout', 'austerity', 'default'
    ],
    'security_crisis': [
        'terrorism', 'attack', 'shooting', 'bombing', 'hostage', 'threat',
        'security breach', 'cyberattack', 'war', 'conflict', 'invasion',
        'military action', 'armed conflict', 'national security', 'terrorist',
        'explosion', 'assault', 'casualties', 'militant'
    ],
    'environmental_crisis': [
        'climate crisis', 'pollution', 'oil spill', 'toxic', 'contamination',
        'environmental disaster', 'radiation', 'chemical leak', 'nuclear accident',
        'ecological collapse', 'mass extinction', 'deforestation', 'global warming',
        'greenhouse gas', 'environmental damage'
    ]
}

print("Crisis categories defined:")
for category, keywords in CRISIS_CATEGORIES.items():
    print(f"  {category}: {len(keywords)} keywords")

Crisis categories defined:
  health_crisis: 21 keywords
  natural_disaster: 20 keywords
  political_crisis: 19 keywords
  economic_crisis: 17 keywords
  security_crisis: 19 keywords
  environmental_crisis: 15 keywords


In [27]:
def detect_crisis_in_text(text, crisis_categories):

    text_lower = text.lower()
    detected_categories = []
    matched_keywords = {}
    total_matches = 0

    for category, keywords in crisis_categories.items():
        matches = [kw for kw in keywords if kw in text_lower]
        if matches:
            detected_categories.append(category)
            matched_keywords[category] = matches
            total_matches += len(matches)

    return detected_categories, matched_keywords, total_matches

print("Crisis detection function defined!")

Crisis detection function defined!


In [29]:
def test_article(title, text, model, vectorizer, crisis=False):
    content = title + " " + text
    content_tfidf = vectorizer.transform([content])

    probs = model.predict_proba(content_tfidf)[0]
    class_to_prob = dict(zip(model.classes_, probs))

    prob_fake = class_to_prob.get(0, None)
    prob_real = class_to_prob.get(1, None)
    if prob_fake is None or prob_real is None:
        raise ValueError(f"Expected classes [0, 1], but got {model.classes_}")

    threshold = 0.8 if crisis else 0.5
    classification = "Most likely real" if prob_real >= threshold else "Most likely fake"

    # Crisis detection
    detected_categories, matched_keywords, crisis_intensity = detect_crisis_in_text(
        content, CRISIS_CATEGORIES
    )

    # Print crisis info if any detected
    if detected_categories:
        print("CRISIS CATEGORIES DETECTED:")
        for category in detected_categories:
            keywords = matched_keywords[category]
            print(f"  • {category.replace('_', ' ').title()}: {', '.join(keywords)}")
        print(f"  Total crisis keywords: {crisis_intensity}\n")

    return classification, prob_real, prob_fake

title = "Coronavirus, 100 days that changed the world"
text = "turbulent decade had reached its final day. It was New Year's Eve 2019 and much of the world was preparing to celebrate. The obituaries of the 2010s had dwelt on eruptions and waves that would shape the era ahead: Brexit, the Syrian civil war, refugee crises, social media proliferation, and nationalism roaring back to life. They were written too soon. It was not until these last hours, before the toasts and countdowns had commenced, that the decade's most consequential development of all broke the surface. At 1.38pm on 31 December, a Chinese government website announced the detection of a pneumonia of unknown cause in the area surrounding the South China seafood wholesale market in Wuhan, an industrial city of 11 million people. The outbreak was one of at least a dozen to be confirmed by the World Health Organization that December, including cases of Ebola in west Africa, measles in the Pacific and dengue fever in Afghanistan. Outside China, its discovery was barely noticed. Over the next 100 days, the virus would freeze international travel, extinguish economic activity and confine half of humanity to their homes, infecting more than a million people and counting, including an Iranian vice-president, the actor Idris Elba, and the British prime minister. By the middle of April, more than 75,000 would be dead. But all that was still unimaginable at the end of December, as 11.59pm ticked over to midnight, fireworks exploded and people embraced at parties and in packed streets."

result, prob_real, prob_fake = test_article(
    title=title,
    text=text,
    model=svm,
    vectorizer=vectorizer,
    crisis=True
)

print("Classification:", result)
print("Probability real:", round(prob_real, 3))
print("Probability fake:", round(prob_fake, 3))

CRISIS CATEGORIES DETECTED:
  • Health Crisis: outbreak, virus, ebola
  • Security Crisis: war
  Total crisis keywords: 4

Classification: Most likely fake
Probability real: 0.717
Probability fake: 0.283


In [30]:
# Analyze crisis patterns across the test set
print("="*70)
print("ANALYZING CRISIS PATTERNS IN TEST SET")
print("="*70)

crisis_stats = {cat: 0 for cat in CRISIS_CATEGORIES.keys()}
total_crisis_articles = 0

print("\nProcessing test set...")
for content in X_test:
    detected_cats, matched_kw, intensity = detect_crisis_in_text(content, CRISIS_CATEGORIES)

    if detected_cats:
        total_crisis_articles += 1
        for cat in detected_cats:
            crisis_stats[cat] += 1

print(f"\nCRISIS STATISTICS")
print(f"Total test articles: {len(X_test)}")
print(f"Crisis-related articles: {total_crisis_articles} ({total_crisis_articles/len(X_test)*100:.1f}%)")

print(f"\nCRISIS CATEGORY DISTRIBUTION")
sorted_categories = sorted(crisis_stats.items(), key=lambda x: x[1], reverse=True)
for category, count in sorted_categories:
    percentage = count / len(X_test) * 100
    print(f"{category.replace('_', ' ').title():.<30} {count:>5} articles ({percentage:.1f}%)")

ANALYZING CRISIS PATTERNS IN TEST SET

Processing test set...

CRISIS STATISTICS
Total test articles: 14308
Crisis-related articles: 10780 (75.3%)

CRISIS CATEGORY DISTRIBUTION
Security Crisis...............  9127 articles (63.8%)
Political Crisis..............  4090 articles (28.6%)
Health Crisis.................  2281 articles (15.9%)
Natural Disaster..............  1036 articles (7.2%)
Economic Crisis...............   948 articles (6.6%)
Environmental Crisis..........   438 articles (3.1%)
